# 01 · Train one architecture
Set `ARCH` and run all. **If the session disconnects, re-run all cells.** The trainer finds
`RUNS_DIR/<run_name>/latest.pt` and resumes at the exact step (the data order is a function of the step).

The trainer uses the configured effective batch, the WSD learning-rate schedule and validation every
`eval_every` steps. It saves resumable `latest.pt`, validation-selected `best.pt`, and final weights.


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # must be set before torch initialises CUDA


In [ ]:
# ---- paths + locate the code (lm_lib/ and models/) ----
# Put the code on Drive in ONE of these ways:
#   (a) upload the folders lm_lib/ and models/ (+ hf_token.txt) into MyDrive/cs781_capstone/   [or any folder]
#   (b) upload cs781_code.zip (made by make_code_zip.py) anywhere in MyDrive -> it is unzipped automatically
from google.colab import drive
drive.mount('/content/drive')
import sys, os, glob, zipfile, importlib, shutil, time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; must precede CUDA init
MYDRIVE    = '/content/drive/MyDrive'
DRIVE_ROOT = f'{MYDRIVE}/cs781'            # data + runs live here
DRIVE_DATA = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
LOCAL_DATA = '/content/data'

def find_code_dir():
    levels = ['', '*/', '*/*/', '*/*/*/']
    # 1) a folder on Drive containing lm_lib/ (newest wins if there are several copies)
    hits = [h for lv in levels for h in glob.glob(f'{MYDRIVE}/{lv}lm_lib/trainer.py')]
    if hits:
        return os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
    # 2) the newest cs781_code*.zip on Drive -> always re-extracted fresh (so an updated zip replaces old code)
    zips = [z for lv in levels for z in glob.glob(f'{MYDRIVE}/{lv}cs781_code*.zip')]
    if zips:
        z = max(zips, key=os.path.getmtime)
        shutil.rmtree('/content/cs781_code', ignore_errors=True)
        with zipfile.ZipFile(z) as f:
            f.extractall('/content/cs781_code')
        print('unzipped', z, '| zip modified', time.ctime(os.path.getmtime(z)))
        return '/content/cs781_code'
    raise FileNotFoundError('Could not find lm_lib/ on Drive. Upload the lm_lib/ and models/ folders '
                            '(e.g. to MyDrive/cs781_capstone/) or upload cs781_code.zip to MyDrive, then re-run.')

CODE_DIR = find_code_dir()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
importlib.invalidate_caches()
# if lm_lib/models were imported earlier in this kernel, drop them so the fresh code is used
for _m in [m for m in list(sys.modules) if m == 'lm_lib' or m.startswith(('lm_lib.', 'models.')) or m == 'models']:
    del sys.modules[_m]
print('CODE_DIR =', CODE_DIR, '| contents:', sorted(os.listdir(CODE_DIR)))
os.makedirs(DRIVE_DATA, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
ARCH = 'A_qwen_dense'      # A_qwen_dense | B_encdec | C_gdn_hybrid | D_mla_moe
TRAIN_BIN = 'train.bin'    # 'train.bin' = 10% sample; 'train_full.bin' = all data (made by 00b_full_data)
RUN_NAME = ARCH if TRAIN_BIN == 'train.bin' else ARCH + '_full'
!pip -q install tokenizers
if ARCH == 'C_gdn_hybrid':
    !pip -q install flash-linear-attention

In [ ]:
# copy tokenised data from Drive to fast local disk (once per runtime); the training file is TRAIN_BIN if set
import shutil, os
os.makedirs(LOCAL_DATA, exist_ok=True)
_train_files = [globals()['TRAIN_BIN']] if 'TRAIN_BIN' in globals() else []
for f in _train_files + ['val.bin', 'tokenizer.json', 'val_prompts.json', 'val_refs.json', 'val_srcs.json', 'test.json']:
    src, dst = f'{DRIVE_DATA}/{f}', f'{LOCAL_DATA}/{f}'
    if os.path.exists(src) and (not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src)):
        shutil.copyfile(src, dst)
!ls -lh {LOCAL_DATA}

In [ ]:
from lm_lib.config import TrainConfig
from lm_lib.trainer import train
from lm_lib import utils

cfg = TrainConfig(
    arch=ARCH,
    run_name=RUN_NAME,             # change to start a fresh run (e.g. ARCH + '_v2'); same name => resume
    data_dir=LOCAL_DATA,
    out_dir=RUNS_DIR,
    train_bin=TRAIN_BIN,
    init_from='',                  # e.g. f'{RUNS_DIR}/A_qwen_dense/best.pt' to start from the 10% model's weights
    seq_len=1024, src_len=768,     # src_len: encoder length for B; also where the tail loss starts
    global_batch_seqs=256,         # ~262k tokens / step
    micro_batch=0,                 # 0 => auto-calibrate
    mem_frac=0.90,
    train_tokens=0,                # 0 => one pass over train.bin (the 10% sample)
    lr=3e-4, warmup_frac=0.02, decay_frac=0.2, min_lr_ratio=0.1, weight_decay=0.1,
    compile=(ARCH != 'D_mla_moe'), # MoE routing has data-dependent shapes (recompiles every step); try C compiled, set False if it errors
    eval_every=200, ckpt_minutes=15, max_minutes=0,
    # full-data run: keep weights-only snapshots every 2000 steps (~1.1 GB each, for checkpoint comparison) and a
    # resumable copy at the end of the WSD stable phase (to extend training later without re-warming)
    snapshot_every=2000 if TRAIN_BIN == 'train_full.bin' else 0,
    keep_stable_ckpt=(TRAIN_BIN == 'train_full.bin'),
    extra={},                      # forwarded to build_model(**extra)
)
print(cfg.to_json())

In [ ]:
# pre-flight checks before a long run: param budget, shapes, init loss ~ ln(V), causality,
# gradient flow, one-batch overfit, encoder wiring / KV-cache consistency. Fix every FAIL first.
from tokenizers import Tokenizer
from lm_lib.model_check import check_model
V = Tokenizer.from_file(f'{LOCAL_DATA}/tokenizer.json').get_vocab_size()
checks = check_model(ARCH, V, cfg.seq_len, cfg.src_len, cfg.extra)
assert all(checks.values()), 'fix the failing checks before training'

In [ ]:
result = train(cfg)
result

In [ ]:
# training curves (ancillary plotting)
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f'{RUNS_DIR}/{cfg.run_name}/log.csv')
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(log.tokens / 1e6, log.loss, label='train'); v = log.dropna(subset=['val_loss'])
ax[0].plot(v.tokens / 1e6, v.val_loss, 'o-', label='val'); ax[0].set_xlabel('M tokens'); ax[0].legend(); ax[0].set_title('loss')
ax[1].plot(log.step, log.lr); ax[1].set_title('lr')
ax[2].plot(log.step, log.mfu * 100); ax[2].set_title('MFU %')
plt.show()